# Stage 1

In [8]:
import os
import sys
import gymnasium as gym
import torch

sys.path.insert(0, os.path.abspath(".."))

from config.match_config import MatchConfig, PlayerSlot, PlayerStats
from src.engine.modes.drill_mode import SoloDrillMode
from src.rl.env_wrapper import HaxballGymEnv
from src.rl.ppo_core import ActorCritic
from src.rl.reset_strategies import ProximalStrikerReset
from src.rl.reward_shapers import Stage1Reward
from src.rl.trainer import train_ppo_vectorized

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def make_env():
    match_cfg = MatchConfig(
        mode=SoloDrillMode(),
        roster=[PlayerSlot(team="red", stats=PlayerStats(name="Agent"))],
    )
    return HaxballGymEnv(
        match_config=match_cfg,
        reward_shaper=Stage1Reward(),
        reset_strategy=ProximalStrikerReset(),
        max_steps=300,
    )


# 1. Parallel environments for training exploration
num_envs = 16
train_envs = gym.vector.AsyncVectorEnv(
    [make_env for _ in range(num_envs)]
)

# 2. Dedicated single environment for fixed-seed validation
eval_env = make_env()

# 3. Model setup
model = ActorCritic(obs_dim=80).to(device)



In [9]:
# 4. Train with 50-episode deterministic evaluation every 100,000 steps
train_ppo_vectorized(
    envs=train_envs,
    eval_env=eval_env,
    model=model,
    device=device,
    total_timesteps=100_000,
    num_envs=num_envs,
    eval_freq=100_000,
    eval_episodes=100,
    save_dir="models/stage1",
    lr_initial=3e-4,
    lr_final=1e-5,
)

🚀 Training (80 dims) | Benchmark every 100000 steps...

📊 [EVALUATION @ Step  102400] Scored:  10.0% (10/100) | Conceded:   0.0% (0/100) | Net: +10 | Touch:  68.0% | Avg Steps: 283.3 | Mean Reward: -36.37
   ⭐ New verified best model saved: models/stage1/best_stage2.pt
      [Net: +10 | Scored: 10.0% | Reward: -36.37 | Speed: 283.3 steps]

✅ Training completed. Final model saved to models/stage1/final_stage2.pt


# Testing

In [ ]:
import torch
from src.rl.ppo_core import ActorCritic
from src.rl.benchmarker import RLController, run_arena, run_solo_drill, render_match
from config.match_config import PlayerSlot, PlayerStats
from src.bots.heuristic_bot import TeamHeuristicCoordinator
from src.engine.controllers import HeuristicBotController

device = torch.device("cpu") # Fast inference on CPU

# 1. Load your RL Models
obs_dim = 80
stage1_model = ActorCritic(obs_dim).to(device)
stage1_model.load_state_dict(torch.load("models/stage1/best_stage1.pt", map_location=device))


# 2. Setup Team Coordinators
red_rl_controller = RLController(stage1_model, team="red", device=device)

red_heuristic_coord = TeamHeuristicCoordinator(team="red")
red_heuristic_controller = HeuristicBotController(red_heuristic_coord)

blue_heuristic_coord = TeamHeuristicCoordinator(team="blue")
blue_heuristic_controller = HeuristicBotController(blue_heuristic_coord)

blue_rl_controller = RLController(stage1_model, team="blue")

# ==========================================
# TEST 1: The Diagnostic Solo Drill
# ==========================================
print("--- TEST 1: EMPTY NET DIAGNOSTIC ---")
run_solo_drill(
    agent_roster=[PlayerSlot("red", PlayerStats("RL_Test"), red_rl_controller)],
    num_episodes=5,
    time_limit=60.0
)


--- TEST 1: EMPTY NET DIAGNOSTIC ---
🎯 Running Solo Drill: 20.0s per episode (5 Episodes)
   Ep 1: 0 goals
   Ep 2: 0 goals
   Ep 3: 0 goals
   Ep 4: 0 goals
   Ep 5: 0 goals
📊 Average Scoring Rate: 0.00 goals per 20.0s



0.0

In [17]:
# ==========================================
# TEST 2: The Arena 
# RL Agent (Red) vs Heuristic Bot (Blue)
# ==========================================
print("\n--- TEST 2: THE ARENA (1v1) ---")
red_roster = [PlayerSlot("red", PlayerStats("RL_Agent"), red_rl_controller)]
blue_roster = [PlayerSlot("blue", PlayerStats("Bot"), blue_rl_controller)]

stats = run_arena(
    red_roster=red_roster,
    blue_roster=blue_roster,
    num_matches=1,
    time_limit=60.0,  # 60 second matches
    score_limit=3
)




--- TEST 2: THE ARENA (1v1) ---
🏟️ Running Arena: 1 RED vs 1 BLUE (1 Matches)


KeyboardInterrupt: 

In [ ]:
# ==========================================
# TEST 3: Kaggle-Style Visualization
# Watch the matchup in HTML format
# ==========================================
red_roster = [PlayerSlot("red", PlayerStats("RL_Agent"), red_rl_controller)]
blue_roster = [PlayerSlot("blue", PlayerStats("Bot"), blue_heuristic_controller)]


print("\n--- TEST 3: RENDER MATCH ---")
render_match(
    red_roster=red_roster,
    blue_roster=blue_roster,
    num_matches=1,
    time_limit=90.0,
    save_path="renders/arena"
)



--- TEST 3: RENDER MATCH ---
🎬 Generating 1 replays...
Game 1 Result: BLUE WINS! 🎉 (2 - 0)
Total Turns (Steps): 14108
Replay saved to: renders/arena/2026-08-21_21-28-13_match_1.html

